# JParaCrawl export → Kokoro-82M → Whisper QC

`feature/kokoro-tts` 用。元のQwenノートブックとは出力先が別です。T4 GPUランタイムを選択してください。

最初は10件→100件まで。1000件と本番は初期状態で無効です。`すべて実行`でも本番は始まりません。

再接続時は1〜6（Drive・repo・依存・設定・入力検証・GPU）を実行後、statusから再開できます。新しいランタイムでは依存の再導入が必要です。モデルは必要時に再ロードします。同じ出力先を複数ランタイムから同時に実行しないでください。


## 1. Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2. リポジトリを復元

既存Qwenのcheckoutには触れず、専用ディレクトリを使用します。追跡ファイルに変更があれば停止します。


In [ ]:
import os, sys, subprocess, json
from pathlib import Path
REPO_DIR = Path('/content/ja-en-direct-s2st-corpus-kokoro')
BRANCH = 'feature/kokoro-tts'
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, 'https://github.com/Yaaamashiro/ja-en-direct-s2st-corpus.git', str(REPO_DIR)], check=True)
else:
    dirty = subprocess.check_output(['git', 'status', '--porcelain', '--untracked-files=no'], cwd=REPO_DIR, text=True)
    assert not dirty.strip(), '未コミット変更があります。保存してから再開してください。'
    subprocess.run(['git', 'fetch', 'origin', BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'switch', BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'merge', '--ff-only', 'origin/' + BRANCH], cwd=REPO_DIR, check=True)
os.chdir(REPO_DIR)
print(subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())


## 3. 依存環境

ノートブック本体のPythonと独立したPython 3.12環境を使います。Qwen・flash-attnは不要です。モデル/G2Pの初回ダウンロードがあります。


In [ ]:
subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run(['apt-get', 'install', '-y', 'espeak-ng', 'build-essential', 'cmake'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'uv==0.8.22'], check=True)
UV = [sys.executable, '-m', 'uv']
subprocess.run(UV + ['python', 'install', '3.12'], check=True)
ENV_DIR = Path('/content/s2st-kokoro-py312')
if not ENV_DIR.exists():
    subprocess.run(UV + ['venv', '--python', '3.12', '--seed', str(ENV_DIR)], check=True)
PYTHON = str(ENV_DIR / 'bin/python')
subprocess.run([PYTHON, '-m', 'pip', 'install', 'torch==2.7.1', 'torchaudio==2.7.1', '--index-url', 'https://download.pytorch.org/whl/cu126'], check=True)
subprocess.run([PYTHON, '-m', 'pip', 'install', '-r', 'requirements/kokoro-colab.txt'], check=True)
subprocess.run([PYTHON, '-m', 'pip', 'install', '-e', '.', '--no-deps'], check=True)
subprocess.run([PYTHON, '-m', 'pip', 'check'], check=True)
# UniDicパッケージだけでは辞書本体は入らないため、版を指定して取得
dictionary_check = "from pathlib import Path; import unidic; p=Path(unidic.DICDIR); assert (p/'sys.dic').is_file() and (p/'version').read_text().strip() == 'unidic-3.1.0+2021-08-31'"
if subprocess.run([PYTHON, '-c', dictionary_check]).returncode:
    subprocess.run([PYTHON, '-m', 'unidic', 'download', '3.1.0+2021-08-31'], check=True)
# Qwen用バッチ設定を引き継がない
os.environ.pop('S2ST_TTS_BATCH_SIZE', None)
os.environ.pop('S2ST_ASR_BATCH_SIZE', None)
os.environ['HF_HOME'] = '/content/hf-kokoro-cache'
os.environ['S2ST_MODEL_ROOT'] = '/content/kokoro-models'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'


## 4. 設定の保存・復元と入力の存在確認

初回のみテンプレートをDriveにコピーします。以後は保存した設定を再利用します。入力パス・voiceを変える場合は、この設定ファイルを編集してください。生成開始後のvoice等の変更は拒否されます。変更した音声を試す場合はsmokeの出力先も新しくしてください。

入力がない場合は、先に `jparacrawl-filtering/scripts/export_text_corpus.py` をそちらのリポジトリで実行してください。このノートブックはrawやparquetを読みません。


In [ ]:
import shutil
DATA_ROOT = Path('/content/drive/MyDrive/ja-en-direct-s2st-corpus')
DATA_ROOT.mkdir(parents=True, exist_ok=True)
CONFIG = DATA_ROOT / 'kokoro-jparacrawl-config.yaml'
if not CONFIG.exists():
    shutil.copyfile(REPO_DIR / 'configs/production-kokoro-jparacrawl.yaml', CONFIG)
print(CONFIG.read_text())
# 保存済み設定からパスを取得（ホストPythonのyamlには依存しない）
def read_settings():
    script = 'import yaml,json,sys; print(json.dumps(yaml.safe_load(open(sys.argv[1]))))'
    return json.loads(subprocess.check_output([PYTHON, '-c', script, str(CONFIG)], text=True))
settings = read_settings()
for key in ('manifest', 'summary'):
    path = Path(settings['prepare'][key])
    assert path.is_file(), f'export入力がありません: {path}'
    print(key, path, path.stat().st_size)
# 実環境の依存・実装commitも保存
ENV_REPORT = DATA_ROOT / 'kokoro-environment'
ENV_REPORT.mkdir(exist_ok=True)
(ENV_REPORT / 'pip-freeze.txt').write_text(subprocess.check_output([PYTHON, '-m', 'pip', 'freeze'], text=True))
(ENV_REPORT / 'git-commit.txt').write_text(subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True))


## 5. summary・件数・SHA256検証

本番と同じ全入力の検証です。テキストの修正や再選別は行いません。ログはDriveへ保存されます。


In [ ]:
from datetime import datetime
LOG_DIR = DATA_ROOT / 'kokoro-logs'
LOG_DIR.mkdir(exist_ok=True)
def run_command(command, *args, production=False):
    cmd = [PYTHON, '-u', '-m', 's2st_corpus.cli', '--config', str(CONFIG)]
    if production:
        cmd.append('--allow-production')
    cmd.extend([command, *map(str, args)])
    log_path = LOG_DIR / (datetime.now().strftime('%Y%m%d-%H%M%S-%f') + '-' + command + '.log')
    print('Log:', log_path, flush=True)
    with log_path.open('w', encoding='utf-8') as log:
        with subprocess.Popen(cmd, cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                              text=True, encoding='utf-8', errors='replace', bufsize=1) as process:
            try:
                for line in process.stdout:
                    print(line, end='', flush=True)
                    log.write(line)
                    log.flush()
                rc = process.wait()
            except BaseException:
                process.terminate()
                try:
                    process.wait(timeout=10)
                except subprocess.TimeoutExpired:
                    process.kill()
                    process.wait()
                raise
    if rc:
        raise RuntimeError(f'{command} failed ({rc}): {log_path}')
run_command('prepare')


## 6. GPU・容量確認

Kokoroはfloat32、WhisperはGPU上でfloat16。T4にBF16は要求しません。GPU使用量が少ないことだけでバッチを上げないでください。Kokoro公式pipelineは逐次生成し、QCのみASRバッチを使います。


In [ ]:
run_command('preflight')


## 7. Kokoro / G2P preload

固定revisionのモデル・voiceを取得します。このセルは生成せず終了します。生成セルではモデルを再ロードします。


In [ ]:
run_command('preload')


## 8. 固定文でvoice試聴（任意）

日本語5種類、英語は指定した候補。ここで試聴しても本番voiceは変わりません。採用voiceは設定で確定してからsmokeへ進んでください。


In [ ]:
RUN_VOICE_SAMPLES = False
VOICE_DIR = DATA_ROOT / 'kokoro-voice-samples'
if RUN_VOICE_SAMPLES:
    run_command('voice-samples', '--output-dir', VOICE_DIR, '--en-voices', 'af_heart', 'af_bella', 'am_adam')
    from IPython.display import Audio, display
    for path in sorted(VOICE_DIR.glob('*.wav')):
        print(path.name)
        display(Audio(filename=str(path)))


## 9. 10件 → 100件 full TTS + Whisper QC

export順の先頭N件を使用します。各件数の出力は独立します。QC落ちも結果として記録されます。`returncode=0`だけで全件合格とは限りません。

再接続時に既存のsmokeを再実行する必要はありません。


In [ ]:
RUN_SMOKE_10 = True
if RUN_SMOKE_10:
    run_command('smoke-test', '--pairs', 10)


In [ ]:
RUN_SMOKE_100 = True
if RUN_SMOKE_100:
    run_command('smoke-test', '--pairs', 100)


## 10. 任意の1000件benchmark

CUを使います。100件の品質・RTF・generation failuresを確認してから有効化してください。未完了セッションを含む場合、full wall timeは過小評価されることがあります。


In [ ]:
RUN_BENCHMARK_1000 = False
if RUN_BENCHMARK_1000:
    run_command('smoke-test', '--pairs', 1000)


## 11. 計画 / 12. 状態

`expected_hours_per_language`は容量見積もりです。入力30万件が必ず300時間になる意味ではありません。statusはmanifest状態を示します。consolidate時にWAVのSHA256も検証します。


In [ ]:
run_command('plan')
run_command('status')


## ここで停止してsmoke結果を確認

初期状態では下の本番セルは何も生成しません。音質・CER/WER・RTF・容量を確認し、本番実行を決めた後だけ変更してください。再開にも同じ設定・出力先を使います。


In [ ]:
APPROVE_PRODUCTION = False  # 結果を確認してから手動でTrue
RUN_ALL_REMAINING = False


## 13. 次の1 shard


In [ ]:
if APPROVE_PRODUCTION and not RUN_ALL_REMAINING:
    run_command('run-next-shard', production=True)
else:
    print('本番1 shardは実行していません。')


## 14. 残り全shard


In [ ]:
if APPROVE_PRODUCTION and RUN_ALL_REMAINING:
    run_command('run-remaining-shards', production=True)
else:
    print('残り全shardは実行していません。')


## 15. 統合

全shardが揃っていない場合はエラーにします。途中集計だけ必要な場合は手動で `--allow-incomplete` を指定します。

保存先：本番 `output_dir/manifests/releases/{accepted,all}.jsonl` と `summary.json`。学習にはacceptedを使います。試行音声が全て合格しているわけではありません。


In [ ]:
if APPROVE_PRODUCTION:
    run_command('consolidate')
else:
    print('本番統合は実行していません。smoke集計は各pairs-N/production/manifests/releases/summary.jsonにあります。')
